# Hands-on: Transformers on DNA and Proteins

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lobaton/AI-Workshop-UNTRM/blob/main/session2_inside_the_black_box/src/Attention_colab.ipynb)

This notebook lets you **train a tiny transformer** that reads DNA and compare it with a CNN, look at its
**attention maps**, and then run a real **protein language model (ESM-2)** on any protein you choose. The ideas
behind it (attention, positional encoding, tokens) are explained in the presenter's [marimo notebook](https://molab.marimo.io/github/lobaton/AI-Workshop-UNTRM/blob/main/session2_inside_the_black_box/src/Attention.py).

**No coding needed:** run each cell in order with the ▶ button (or `Shift + Enter`). Cells marked **✏️ Settings**
have a few values you can change before running them. Everything runs on CPU; a GPU speeds up the protein part.

## 1. Setup
Downloads the shared workshop code.

In [ ]:
import os, urllib.request
if not os.path.exists("attention_common.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/lobaton/AI-Workshop-UNTRM/main/session2_inside_the_black_box/src/attention_common.py",
                               "attention_common.py")

import numpy as np
import matplotlib.pyplot as plt
import torch
import attention_common as at

plt.rcParams.update({"font.size": 14, "lines.linewidth": 3})
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", "GPU" if device == "cuda" else "CPU")

## 2. ✏️ Settings: the DNA task

Every sequence contains motif A and motif B once. The label says whether A comes **before** B (as the -35 and
-10 boxes do in a bacterial promoter) or after it. Only the **order** tells the classes apart.

- `MOTIF_A`, `MOTIF_B`: the two motifs (letters A, C, G, T, both 6 letters long).
- `SEQ_LEN`: length of each sequence. `MIN_GAP`: minimum number of letters between the two motifs.
  `MAX_DIST`: maximum distance between the starts of the two motifs (at most `SEQ_LEN - 6`).
- `MUTATION_RATE`: chance that each letter of a planted motif is changed to a random letter.
- `USE_POSITION`: add positional encoding (`True`) or not (`False`). `N_HEADS`: attention heads (1, 2 or 4).
- `EPOCHS`: passes through the 3,000 training sequences.

In [ ]:
MOTIF_A = "TTGACA"
MOTIF_B = "TATAAT"
SEQ_LEN = 64
MIN_GAP = 10
MAX_DIST = 30
MUTATION_RATE = 0.0
USE_POSITION = True
N_HEADS = 2
EPOCHS = 15

## 3. Create the data
Two examples, with the motifs in lowercase.

In [ ]:
settings = dict(length=SEQ_LEN, min_gap=MIN_GAP, max_dist=MAX_DIST, mutation=MUTATION_RATE,
                motif_a=MOTIF_A, motif_b=MOTIF_B)
data = at.make_dataset(**settings)
(x_train, y_train, _), (x_val, y_val, _), (x_test, y_test, s_test) = data["train"], data["validation"], data["test"]

def show(i):
    text = list(at.to_text(x_test[i]))
    for p in s_test[i]:
        text[p:p + 6] = [c.lower() for c in text[p:p + 6]]
    print(("A before B: " if y_test[i] else "B before A: ") + "".join(text))

show(int((y_test == 1).argmax()))
show(int((y_test == 0).argmax()))

## 4. Train the CNN and the transformer

The CNN is the one from the DNA motif notebook (filters + global max pooling). The transformer reads the DNA as
overlapping 6-letter tokens and has one attention layer.

In [ ]:
torch.manual_seed(0)
cnn = at.MotifCNN()
cnn_history = at.train_model(cnn, x_train, y_train, x_val, y_val, epochs=EPOCHS, lr=1e-2)

torch.manual_seed(0)
transformer = at.TinyTransformer(length=SEQ_LEN, heads=N_HEADS, use_position=USE_POSITION)
tt_history = at.train_model(transformer, x_train, y_train, x_val, y_val, epochs=EPOCHS, lr=3e-3,
                            log=print)

print(f"\nTest accuracy (1,000 new sequences)   chance: 50%")
print(f"  1D CNN:      {at.accuracy(cnn, x_test, y_test):.1%}")
print(f"  Transformer: {at.accuracy(transformer, x_test, y_test):.1%}")

plt.figure(figsize=(9, 4))
plt.plot(np.arange(1, EPOCHS + 1), tt_history["val_acc"], "o-", color="#0072B2", label="Transformer")
plt.plot(np.arange(1, EPOCHS + 1), cnn_history["val_acc"], "o-", color="#E69F00", label="1D CNN")
plt.axhline(0.5, color="black", linestyle="--", linewidth=1.5, label="Chance")
plt.xlabel("Epoch"); plt.ylabel("Validation accuracy"); plt.ylim(0.4, 1.02); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Try:**
- `USE_POSITION = False`: can the transformer still tell the order?
- `MUTATION_RATE = 0.1`: motifs with mistakes, as in real DNA. A mutated motif becomes a different 6-mer token.
- `SEQ_LEN = 120` and `MAX_DIST = 100`: motifs much farther apart.
- `N_HEADS = 1`: does one head still learn the task? Look at the attention maps below.

## 5. Attention maps

For one test sequence: where does each token look, for every head? The orange and blue lines mark the two
motif tokens. ✏️ Change `EXAMPLE` to look at other sequences.

In [ ]:
EXAMPLE = 0

weights = at.attention_maps(transformer, x_test[EXAMPLE])  # (heads, tokens, tokens)
pa, pb = s_test[EXAMPLE]
fig, axes = plt.subplots(1, len(weights), figsize=(6.5 * len(weights), 6), squeeze=False)
for h, ax in enumerate(axes[0]):
    ax.imshow(weights[h], cmap="Greys", vmin=0, vmax=weights[h].max())
    ax.axvline(pa, color="#E69F00"); ax.axhline(pa, color="#E69F00")
    ax.axvline(pb, color="#0072B2"); ax.axhline(pb, color="#0072B2")
    ax.set_title(f"Head {h + 1}: {weights[h][:, pa].mean():.0%} on A, {weights[h][:, pb].mean():.0%} on B")
    ax.set_xlabel("Token being looked at"); ax.set_ylabel("Token that looks")
plt.tight_layout(); plt.show()
print("Truth:", "A before B" if y_test[EXAMPLE] else "B before A",
      f"| transformer: {at.predict(transformer, x_test[EXAMPLE:EXAMPLE + 1])[0]:.0%} sure it is A before B")

## 6. ✏️ Your own protein with ESM-2

**ESM-2** is a transformer trained on millions of protein sequences by hiding amino acids and filling them in.
Paste any protein sequence (one-letter amino-acid code, for example from [UniProt](https://www.uniprot.org)).
The default is **ubiquitin**, a small protein that is almost identical in plants and animals.

- `ESM_MODEL`: `"facebook/esm2_t6_8M_UR50D"` (8M parameters, fast), `"facebook/esm2_t12_35M_UR50D"` (35M) or
  `"facebook/esm2_t30_150M_UR50D"` (150M, about 600 MB to download). Bigger models predict contacts better.
- `LAYER`, `HEAD`: which attention map to look at in detail (layers and heads are numbered from 1).

In [ ]:
PROTEIN = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
ESM_MODEL = "facebook/esm2_t6_8M_UR50D"
LAYER = 6
HEAD = 5

In [ ]:
from transformers import AutoTokenizer, EsmForMaskedLM

tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL)
esm = EsmForMaskedLM.from_pretrained(ESM_MODEL, attn_implementation="eager").to(device).eval()
inputs = tokenizer(PROTEIN, return_tensors="pt").to(device)
with torch.no_grad():
    out = esm(**inputs, output_attentions=True)
    contacts = esm.predict_contacts(inputs["input_ids"], inputs["attention_mask"])[0].cpu().numpy()
# (layers, heads, amino acids, amino acids), without the start/end tokens
attention = torch.stack(out.attentions)[:, 0, :, 1:-1, 1:-1].cpu().numpy()
n_layers, n_heads = attention.shape[:2]
print(f"{len(PROTEIN)} amino acids, {sum(p.numel() for p in esm.parameters()):,} parameters, "
      f"{n_layers} layers × {n_heads} heads")

# All heads of the chosen layer, small
cols = 10
rows = int(np.ceil(n_heads / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2.1 * rows), squeeze=False)
for h, ax in enumerate(axes.flat):
    ax.axis("off")
    if h < n_heads:
        a = attention[LAYER - 1, h]
        ax.imshow(a, cmap="Greys", vmax=np.percentile(a, 99))
        ax.set_title(f"head {h + 1}", fontsize=11)
fig.suptitle(f"Layer {LAYER}: every head looks at something different")
plt.tight_layout(); plt.show()

# The chosen head and the predicted contact map
fig, axes = plt.subplots(1, 2, figsize=(14, 6.3))
a = attention[LAYER - 1, HEAD - 1]
axes[0].imshow((a + a.T) / 2, cmap="Greys", vmax=np.percentile(a, 99))
axes[0].set_title(f"Attention, layer {LAYER}, head {HEAD}")
axes[1].imshow(contacts, cmap="Greys")
axes[1].set_title("Predicted contacts (all heads combined)")
for ax in axes:
    ax.set_xlabel("Amino acid position"); ax.set_ylabel("Amino acid position")
plt.tight_layout(); plt.show()

**Try:** look at several layers: do the early layers mostly look at neighbors (near the diagonal)? Switch
`ESM_MODEL` to the 35M or 150M model and compare the predicted contact maps. Lines parallel to the diagonal
are usually helices; lines perpendicular to it, strands pairing in a sheet.

## Questions for discussion

1. In the DNA task, what happens without positional encoding, and why? Where else in biology does the **order**
   or **spacing** of elements matter?
2. What would a **token** be in your own data (field images, sensor readings, sequences)? Is there a large
   pre-trained model for it that you could adapt instead of training from scratch?